# Example 8: 在远程机器上分发、提交任务

这个例子展示了如何在远程机器上分发并执行 ABAQUS Job, 任何与远程相关的内容均需要在 `BatchAbaqusProcessor` 构造中的 `Hosts` 中显示说明, 否则所有任务都会在本地执行.

- `max_concurrent`: 决定了一台目标机器上可以同时运行多少个任务
- `weight`: 决定了一台目标机器上的任务分配数量的权重

---
关于如何配置目标机器:

```bash
Add-WindowsCapability -Online -Name OpenSSH.Server~~~~0.0.1.0
Start-Service sshd
Set-Service -Name sshd -StartupType Automatic
Get-NetFirewallRule -Name *OpenSSH* | Select Name,Enabled,Profile   # 需对当前活动配置文件启用
netstat -an | findstr ":22"  # 确认同时监听 IPv6（应看到 [::]:22，而不只是 0.0.0.0:22）
Get-ChildItem 'C:\','D:\' -Filter abaqus.bat -Recurse -Depth 5 -ErrorAction SilentlyContinue  # 检查abaqus位置
```

In [1]:
from __future__ import annotations

import os

import numpy as np
from dotenv import load_dotenv

from ABQflow import (
	BatchAbaqusProcessor,
	HostSpec,
	generate_from_array,
	Timeouts,
)
from ABQflow.core.spec import HookSpec, JobSpec, PreparationSpec

%reload_ext autoreload
%autoreload 2

ABAQUS_CAE = 'C:/Applications/SIMULIA/Commands/2026/abaqus.bat'
CWD = os.getcwd()
OUTPUT_DIR = os.path.join(CWD, "examples/08_RemoteSubmission/output")

## 设置远程目标机器

In [2]:
load_dotenv(r"examples\08_RemoteSubmission\password.env")

HOSTS = [
	HostSpec.local(
		name='DESKTOP-Millana',
		max_concurrent=1,
	),
	HostSpec(
		name='SJTU302',
		hostname='SJTU302',                 # bare IPv6/IPv4 literal also works,
		username='SJTU302',                 # with no square brackets
		password=os.getenv("SJTU302_password"),
		abaqus_exe=r'C:\Program Files\SIMULIA\Commands\abaqus.bat',
		work_root = r'D:\LiKaiwen\abqflow_test',
		cpus_per_job=16,
		max_concurrent=1,                   # two jobs at a time on this machine
	),
	HostSpec(
		name='SJTU101',
		hostname='SJTU101',
		username='SJTU101',
		password=os.getenv("SJTU101_password"),
		abaqus_exe=r'C:\SIMULIA\Commands\abaqus.bat',
		work_root=r'D:\LiKaiwen\abqflow_test',
		cpus_per_job=16,
		max_concurrent=2,                  # but only one at a time here
		weight=3.0,                        # while still taking a large share
	),
]

In [3]:
param_names = ['youngs_modulus', 'load_magnitude']
param_values = np.array([
	[200000, 2000],
	[210000, 3000],
	[220000, 4000],
	[230000, 5000]
])

base_spec = JobSpec(
	job_name = "remote_submission_example",
	workflow = "modular",
	preparation = PreparationSpec(
		kind='inp_based',
		source_path = "./examples/cae_file/planar_stress_nested_scenario.inp",
	),
	pre_extraction = [
		HookSpec(
			script_path = "./examples/extraction_scripts/get_total_mass.py",
			tasks = [
				{"result_name": "total_mass",},
			]
		)
	],
	post_extraction = [
		HookSpec(
			script_path = "./examples/extraction_scripts/get_max_stress_mises.py",
			tasks = [
				{"result_name": "max_stress_mises",},
				{"result_name": "max_displacement",},
			]
		)
	]
)


specs = generate_from_array(
	base_spec=base_spec,
	param_names=param_names,
	samples_array = param_values,
)

In [4]:
processor = BatchAbaqusProcessor(
	batch_data=specs,
	base_output_dir=OUTPUT_DIR,
	cpus_per_job=2,
	duplicate_mode='overwrite',
	abaqus_exe=ABAQUS_CAE,
	timeout=Timeouts(solver=None, preflight=1800),
	hosts=HOSTS,               # <- the only line that makes this remote
)

## 查看当前分配情况

In [5]:
print("Planned assignment:")
for host_name, jobs in processor.assignment().items():
	print(f"  {host_name}: {len(jobs)} job(s) — {', '.join(jobs)}")

Planned assignment:
  DESKTOP-Millana: 1 job(s) — remote_submission_example_0001
  SJTU302: 1 job(s) — remote_submission_example_0002
  SJTU101: 2 job(s) — remote_submission_example_0003, remote_submission_example_0004


In [6]:
outcomes = processor.run_batch(num_parallel_jobs=4)

[09/06/26 17:52:22] WARNING  Existing job dirs: ['remote_submission_example_0001',                                 
                             'remote_submission_example_0002', 'remote_submission_example_0003',                   
                             'remote_submission_example_0004']

c:\SJTU\Projects_Code\24_Abaqus_Pack\.pixi\envs\default\Lib\site-packages\paramiko\client.py:850: UserWarning: Unknown ssh-ed25519 host key for SJTU302: b'2d12a8c0620460d8b0b7d502a65589d1'
  warnings.warn(
c:\SJTU\Projects_Code\24_Abaqus_Pack\.pixi\envs\default\Lib\site-packages\paramiko\client.py:850: UserWarning: Unknown ssh-ed25519 host key for SJTU101: b'f6311ea2605f6328308e4278cc9e3961'
  warnings.warn(


[09/06/26 17:52:26] WARNING  Host 'SJTU101' oversubscribes its CPUs: 2 concurrent job(s) x 16 cpus = 32 cores      
                             requested, but only 15 usable physical core(s) (16 total - 1 reserved); proceeding    
                             anyway.

Output()

In [7]:
outcomes

[JobOutcome(job_name='remote_submission_example_0002', status='COMPLETED', results={'total_mass': 0.00032066262255247625, 'max_stress_mises': 6787.890625, 'max_displacement': 5.984342098236084}, error=None, diagnostics=None, output_dir='c:\\SJTU\\Projects_Code\\24_Abaqus_Pack\\examples/08_RemoteSubmission/output\\remote_submission_example_0002', phases=[{'phase': 'preparation', 'status': 'PREPARATION_SUCCESS', 'started_at': 1788688346.8815012, 'ended_at': 1788688346.888505, 'duration_s': 0.0070037841796875, 'error': None}, {'phase': 'pre_extraction', 'status': 'EXTRACTION_SUCCESS', 'started_at': 1788688346.8905053, 'ended_at': 1788688348.921037, 'duration_s': 2.030531644821167, 'error': None}, {'phase': 'simulation', 'status': 'SIMULATION_SUCCESS', 'started_at': 1788688348.921037, 'ended_at': 1788688376.5266702, 'duration_s': 27.60563325881958, 'error': None}, {'phase': 'post_extraction', 'status': 'EXTRACTION_SUCCESS', 'started_at': 1788688376.5266702, 'ended_at': 1788688376.9194016, 

In [8]:
print("\nResults:")
failed = 0
for oc in sorted(outcomes, key=lambda o: o.job_name):
	mark = 'OK ' if oc.status == 'COMPLETED' else 'FAIL'
	print(f"  [{mark}] {oc.job_name}: {oc.status} ({oc.duration_s or 0:.0f}s)"
		+ (f" — {oc.error}" if oc.error else ''))
	failed += oc.status != 'COMPLETED'

print(f"\n{len(outcomes) - failed}/{len(outcomes)} completed.")
print(f"Small artifacts (.sta/.msg/.dat) were fetched into {OUTPUT_DIR};")
print("the .odb files stayed on the machines that produced them "
	"(set HostSpec.fetch_odb=True to change that).")


Results:
  [OK ] remote_submission_example_0001: COMPLETED (58s)
  [OK ] remote_submission_example_0002: COMPLETED (30s)
  [OK ] remote_submission_example_0003: COMPLETED (35s)
  [OK ] remote_submission_example_0004: COMPLETED (35s)

4/4 completed.
Small artifacts (.sta/.msg/.dat) were fetched into c:\SJTU\Projects_Code\24_Abaqus_Pack\examples/08_RemoteSubmission/output;
the .odb files stayed on the machines that produced them (set HostSpec.fetch_odb=True to change that).
